Import packages

In [1]:
from __future__ import annotations

import math
import time
from typing import List, Optional, Sequence, Tuple

import torch

import numpy as np
import scipy.linalg as la
from scipy.linalg import sqrtm, eigvalsh

In [2]:
def default_device() -> torch.device:
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")


def _prod(seq) -> int:
    p = 1
    for s in seq:
        p *= int(s)
    return p

def _to_numpy(t: torch.Tensor):
    return t.detach().cpu().numpy()

Metrics

In [3]:
def fidelity(state1, state2):
    """
    Compute the quantum state fidelity between two states.

    Parameters
    ----------
    state1 : ndarray
        State vector (n,) or (n,1), or density matrix (n,n).
    state2 : ndarray
        State vector (n,) or (n,1), or density matrix (n,n).

    Returns
    -------
    float
        Fidelity F(rho1, rho2).
    """

    pure = False

    state1 = np.asarray(state1, dtype=complex)
    state2 = np.asarray(state2, dtype=complex)

    # ----- State 1 -----
    if state1.ndim == 1:
        psi = state1.reshape(-1, 1)
        rho1 = psi @ psi.conj().T
        pure = True

    elif state1.ndim == 2 and state1.shape[1] == 1:
        rho1 = state1 @ state1.conj().T
        pure = True

    elif state1.ndim == 2 and state1.shape[0] == state1.shape[1]:
        rho1 = state1

    else:
        raise ValueError("State 1 is not a vector or density matrix.")

    # ----- State 2 -----
    if state2.ndim == 1:
        psi = state2.reshape(-1, 1)
        rho2 = psi @ psi.conj().T
        pure = True

    elif state2.ndim == 2 and state2.shape[1] == 1:
        rho2 = state2 @ state2.conj().T
        pure = True

    elif state2.ndim == 2 and state2.shape[0] == state2.shape[1]:
        rho2 = state2

    else:
        raise ValueError("State 2 is not a vector or density matrix.")

    # Normalize
    rho1 = rho1 / np.trace(rho1)
    rho2 = rho2 / np.trace(rho2)

    # Hermitian projection
    rho1 = 0.5 * (rho1 + rho1.conj().T)
    rho2 = 0.5 * (rho2 + rho2.conj().T)

    if pure:
        # One state is pure
        val = np.trace(rho1 @ rho2)

    else:
        sqrt_rho1 = sqrtm(rho1)
        A = sqrt_rho1 @ rho2 @ sqrt_rho1

        # Since A should be Hermitian PSD, use eigvalsh
        eigvals = eigvalsh(0.5 * (A + A.conj().T))
        eigvals = np.maximum(eigvals, 0.0)

        val = np.sum(np.sqrt(eigvals)) #** 2

    return float(np.real(val))


def trace_dist(rho1, rho2):
    delta = rho1 - rho2
    delta = 0.5 * (delta + delta.conj().T)

    eigvals = eigvalsh(delta)

    return 0.5 * np.sum(np.abs(eigvals))


def entropy(rho, base=np.e):
    """
    Von Neumann entropy:
        S(rho) = -Tr(rho log rho)

    Parameters
    ----------
    rho : (n,n) complex ndarray
        Density matrix.

    Returns
    -------
    float
        Von Neumann entropy in bits.
    """
    eigs = eigvalsh(0.5*(rho + rho.conj().T))
    eigs = eigs[eigs > 0]

    return -np.sum(eigs * np.log(eigs)) / np.log(base)

TT Utilts

In [4]:
# ===========================================================================
# 1. TT basic routines
# ===========================================================================

def entry(cores: List[torch.Tensor], quant_i: Sequence[int], quant_j: Sequence[int]):
    """Single tensor-train entry T[i_1..i_d, j_1..j_d] via chained 2-D mat-vecs."""
    res = torch.ones((1, 1), dtype=cores[0].dtype, device=cores[0].device)
    for k in range(len(cores)):
        res = res @ cores[k][:, quant_i[k], quant_j[k], :]
    return res.item()


def quantize_index(flat_idx: int, dims: Sequence[int]) -> List[int]:
    """Decompose a flat multi-index into per-site indices (pure Python, no tensors needed)."""
    idx = []
    temp = flat_idx
    for d in reversed(dims):
        idx.append(temp % d)
        temp //= d
    return list(reversed(idx))


def init_ttm_cores(dims_row: Sequence[int], dims_col: Sequence[int], ranks: Sequence[int],
                    scale: float = 1.0, seed: Optional[int] = None,
                    complex_output: bool = False, device=None, dtype=None) -> List[torch.Tensor]:
    """Random TT/MPO cores. RNG is seeded on CPU (for cross-device reproducibility) then
    moved to `device`, since CUDA and CPU RNG streams are not interchangeable."""
    device = device or default_device()
    if complex_output:
        dtype = dtype or torch.complex128
        real_dtype = torch.float64 if dtype == torch.complex128 else torch.float32
    else:
        dtype = dtype or torch.float64
        real_dtype = dtype

    gen = torch.Generator(device="cpu")
    if seed is not None:
        gen.manual_seed(seed)

    cores = []
    for k in range(len(dims_row)):
        shape = (ranks[k], dims_row[k], dims_col[k], ranks[k + 1])
        if complex_output:
            real_part = torch.randn(shape, generator=gen, dtype=real_dtype)
            imag_part = torch.randn(shape, generator=gen, dtype=real_dtype)
            core = torch.complex(real_part, imag_part) * scale
        else:
            core = torch.randn(shape, generator=gen, dtype=real_dtype) * scale
        cores.append(core.to(device=device, dtype=dtype))
    return cores


def get_dimsrank(cores: List[torch.Tensor]) -> Tuple[int, List[int], List[int], List[int]]:
    order = len(cores)
    row_dims = [int(c.shape[1]) for c in cores]
    col_dims = [int(c.shape[2]) for c in cores]
    tt_rank = [int(c.shape[0]) for c in cores] + [int(cores[-1].shape[3])]
    return order, row_dims, col_dims, tt_rank

def ten2cores(x, max_rank=float('inf'), threshold=1e-10):
    """
    Decompose a full tensor into TT cores.

    Args:
        x: torch.Tensor of shape (r1, ..., rN, c1, ..., cN) with 2N dims.
        max_rank: int or float('inf'), maximum bond dimension.
        threshold: relative SVD truncation threshold.
    Returns:
        cores: list of torch.Tensor, each shape (r_i, row_dim[i], col_dim[i], r_i+1) in block‑TT form.
    """
    assert x.ndim % 2 == 0, "Number of dimensions must be even"
    order = x.ndim // 2
    row_dims = x.shape[:order]
    col_dims = x.shape[order:]
    ranks = [1] * (order + 1)
    cores = []

    # Permute to interleave row and column indices:
    # (r1,...,rN, c1,...,cN) -> (r1,c1,r2,c2,...,rN,cN)
    p = [order * j + i for i in range(order) for j in range(2)]
    y = x.permute(p).clone()

    for i in range(order - 1):
        m = ranks[i] * row_dims[i] * col_dims[i]
        # Number of remaining elements
        n = (torch.tensor(row_dims[i+1:]).prod() * torch.tensor(col_dims[i+1:]).prod()).item()
        y = y.reshape(m, n)

        # Truncated SVD
        U, S, Vh = torch.linalg.svd(y, full_matrices=False)

        # Relative threshold truncation
        if threshold != 0.0:
            mask = S / S[0] > threshold
            U = U[:, mask]
            S = S[mask]
            Vh = Vh[mask, :]

        # Max rank truncation
        if max_rank != float('inf'):
            r_new = min(S.numel(), max_rank)
            U = U[:, :r_new]
            S = S[:r_new]
            Vh = Vh[:r_new, :]

        # New bond dimension
        r_i_plus_1 = U.shape[1]
        ranks[i + 1] = r_i_plus_1

        # Form core
        core = U.reshape(ranks[i], row_dims[i], col_dims[i], r_i_plus_1)
        cores.append(core)

        # Prepare residual for next step
        y = torch.diag(S) @ Vh

    last_core = y.reshape(ranks[-2], row_dims[-1], col_dims[-1], 1)
    cores.append(last_core)

    return cores

def cores2ten(ttcores: List[torch.Tensor], matricize: bool = False) -> torch.Tensor:
    order, row_dims, col_dims, tt_rank = get_dimsrank(ttcores)
    if tt_rank[0] != 1 or tt_rank[-1] != 1:
        raise ValueError("The first and last rank have to be 1!")

    full = ttcores[0].reshape(row_dims[0] * col_dims[0], tt_rank[1])
    for i in range(1, order):
        full = full @ ttcores[i].reshape(tt_rank[i], row_dims[i] * col_dims[i] * tt_rank[i + 1])
        full = full.reshape(_prod(row_dims[:i + 1]) * _prod(col_dims[:i + 1]), tt_rank[i + 1])

    p = [None] * (2 * order)
    p[::2] = row_dims
    p[1::2] = col_dims
    q = [2 * i for i in range(order)] + [1 + 2 * i for i in range(order)]
    full = full.reshape(p).permute(q)

    if matricize:
        full = full.reshape(_prod(row_dims), _prod(col_dims))
    return full


def core_mul(core_1: torch.Tensor, core_2: torch.Tensor) -> torch.Tensor:
    """Standard MPO-MPO core contraction: (r1,m,n,r2) x (s1,n,p,s2) -> (r1*s1,m,p,r2*s2).
    Implemented with a single einsum (numerically validated equivalent to the original
    fancy-indexing formulation; this version is also considerably faster)."""
    r1, m, n, r2 = core_1.shape
    s1, n2, p, s2 = core_2.shape
    assert n == n2, "Shared (col/row) physical dimension mismatch in core_mul"
    contraction = torch.einsum('aikb,ckjd->acijbd', core_1, core_2)
    return contraction.reshape(r1 * s1, m, p, r2 * s2)


def ttmulcores(ttcores_a: List[torch.Tensor], ttcores_b: List[torch.Tensor]) -> List[torch.Tensor]:
    order_a, row_a, col_a, rank_a = get_dimsrank(ttcores_a)
    order_b, row_b, col_b, rank_b = get_dimsrank(ttcores_b)
    assert col_a == row_b, "Dimensions do not match"
    return [core_mul(a, b) for a, b in zip(ttcores_a, ttcores_b)]


def cores2vec(ttcores: List[torch.Tensor]) -> torch.Tensor:
    return torch.cat([c.reshape(-1) for c in ttcores])


def vec2cores(vec_cores: torch.Tensor, row_dims: Sequence[int], col_dims: Sequence[int],
              tt_rank: Sequence[int]) -> List[torch.Tensor]:
    cores = []
    pos = 0
    for k in range(len(row_dims)):
        n = tt_rank[k] * row_dims[k] * col_dims[k] * tt_rank[k + 1]
        cores.append(vec_cores[pos:pos + n].reshape(tt_rank[k], row_dims[k], col_dims[k], tt_rank[k + 1]))
        pos += n
    return cores


def copy_cores(ttcores: List[torch.Tensor]) -> List[torch.Tensor]:
    return [c.clone() for c in ttcores]


def conj_cores(ttcores: List[torch.Tensor], overwrite: bool = True) -> List[torch.Tensor]:
    cores = ttcores if overwrite else copy_cores(ttcores)
    return [c.conj() for c in cores]


def transpose_cores(ttcores: List[torch.Tensor], overwrite: bool = True,
                     conjugate: bool = False) -> List[torch.Tensor]:
    cores = ttcores if overwrite else copy_cores(ttcores)
    out = []
    for c in cores:
        if conjugate:
            c = c.conj()
        out.append(c.permute(0, 2, 1, 3))
    return out


def scalar_mul(ttcores: List[torch.Tensor], scalar) -> List[torch.Tensor]:
    new_cores = copy_cores(ttcores)
    new_cores[0] = scalar * new_cores[0]
    return new_cores


def ttsumcores(ttcores_a: List[torch.Tensor], ttcores_b: List[torch.Tensor]) -> List[torch.Tensor]:
    order_a, row_a, col_a, rank_a = get_dimsrank(ttcores_a)
    order_b, row_b, col_b, rank_b = get_dimsrank(ttcores_b)
    assert row_a == row_b and col_a == col_b, "Dimensions do not match"

    device, dtype = ttcores_a[0].device, ttcores_a[0].dtype
    # BUGFIX: original used `rank_a[i] + rank_a[i]` (ignoring ttcores_b's rank entirely).
    # The correct TT-sum rank is the sum of *both* operands' ranks at each cut.
    rank_sum = [1] + [rank_a[i] + rank_b[i] for i in range(1, order_a)] + [1]

    sumcores = []
    for i in range(order_a):
        core = torch.zeros((rank_sum[i], row_a[i], col_a[i], rank_sum[i + 1]), dtype=dtype, device=device)
        core[0:rank_a[i], :, :, 0:rank_a[i + 1]] = ttcores_a[i]
        r1, r2 = rank_sum[i] - rank_b[i], rank_sum[i]
        r3, r4 = rank_sum[i + 1] - rank_b[i + 1], rank_sum[i + 1]
        core[r1:r2, :, :, r3:r4] = ttcores_b[i]
        sumcores.append(core)
    return sumcores


def ttsubcores(ttcores_a: List[torch.Tensor], ttcores_b: List[torch.Tensor]) -> List[torch.Tensor]:
    return ttsumcores(ttcores_a, scalar_mul(ttcores_b, -1))


# ===========================================================================
# 2. Generic MPO orthogonalization (full row x col cores), SVD-based
# ===========================================================================

def block2right(cores: List[torch.Tensor], n: int, tol: float = 1e-10,
                 max_rank: float = math.inf) -> List[torch.Tensor]:
    coren, corenp1 = cores[n], cores[n + 1]
    rL, In, Jn, rR = coren.shape
    rR_chk = corenp1.shape[0]
    assert rR == rR_chk

    M = coren.reshape(rL * In * Jn, rR)
    U, S, Vh = torch.linalg.svd(M, full_matrices=False)

    if tol != 0:
        keep = S > tol * S[0]
        U, S, Vh = U[:, keep], S[keep], Vh[keep, :]
    if max_rank != math.inf:
        r_new = min(S.shape[0], int(max_rank))
        U, S, Vh = U[:, :r_new], S[:r_new], Vh[:r_new, :]
    rank_new = U.shape[1]

    coren_new = U.reshape(rL, In, Jn, rank_new)
    SVh = torch.diag(S.to(Vh.dtype)) @ Vh
    corenp1_new = torch.tensordot(SVh, corenp1, dims=([1], [0]))
    return cores[:n] + [coren_new, corenp1_new] + cores[n + 2:]


def block2left(cores: List[torch.Tensor], n: int, tol: float = 1e-10,
                max_rank: float = math.inf) -> List[torch.Tensor]:
    coren, coren_prev = cores[n], cores[n - 1]
    rL, In, Jn, rR = coren.shape
    rL_chk = coren_prev.shape[3]
    assert rL == rL_chk

    M = coren.reshape(rL, In * Jn * rR)
    U, S, Vh = torch.linalg.svd(M, full_matrices=False)

    if tol != 0:
        keep = S > tol * S[0]
        U, S, Vh = U[:, keep], S[keep], Vh[keep, :]
    if max_rank != math.inf:
        r_new = min(S.shape[0], int(max_rank))
        U, S, Vh = U[:, :r_new], S[:r_new], Vh[:r_new, :]
    rank_new = Vh.shape[0]

    US = U @ torch.diag(S.to(U.dtype))
    coren_prev_new = torch.tensordot(coren_prev, US, dims=([3], [0]))
    coren_new = Vh.reshape(rank_new, In, Jn, rR)
    return cores[:n - 1] + [coren_prev_new, coren_new] + cores[n + 1:]


def ortho_right(cores: List[torch.Tensor], threshold: float = 0.0,
                 max_rank: float = math.inf) -> List[torch.Tensor]:
    cores = copy_cores(cores)
    for i in range(len(cores) - 1, 0, -1):
        cores = block2left(cores, i, tol=threshold, max_rank=max_rank)
    return cores


def ortho_left(cores: List[torch.Tensor], threshold: float = 0.0,
                max_rank: float = math.inf) -> List[torch.Tensor]:
    cores = copy_cores(cores)
    for i in range(len(cores) - 1):
        cores = block2right(cores, i, tol=threshold, max_rank=max_rank)
    return cores


# ===========================================================================
# 3. Block-TT orthogonalization (the "active" col-leg moves with the sweep)
# ===========================================================================

def block2rightbtt(cores: List[torch.Tensor], n: int, tol: float = 1e-10,
                    max_rank: float = math.inf) -> List[torch.Tensor]:
    coren, corenp1 = cores[n], cores[n + 1]
    rL, In, K, rR = coren.shape
    rR_chk, Inext, Jnext, rNext = corenp1.shape
    assert rR == rR_chk
    assert Jnext == 1, "Expected singleton physical column index in non-active core"

    M = coren.reshape(rL * In, K * rR)
    U, S, Vh = torch.linalg.svd(M, full_matrices=False)

    if tol != 0:
        keep = S > tol * S[0]
        U, S, Vh = U[:, keep], S[keep], Vh[keep, :]
    if max_rank != math.inf:
        r_new = min(S.shape[0], int(max_rank))
        U, S, Vh = U[:, :r_new], S[:r_new], Vh[:r_new, :]
    rank_new = U.shape[1]

    coren_new = U.reshape(rL, In, 1, rank_new)
    SVh = torch.diag(S.to(Vh.dtype)) @ Vh
    SVh_sh = SVh.reshape(rank_new, K, rR)
    corenp1_new = torch.einsum('akb,bid->aikd', SVh_sh, corenp1[:, :, 0, :])
    return cores[:n] + [coren_new, corenp1_new] + cores[n + 2:]


def block2leftbtt(cores: List[torch.Tensor], n: int, tol: float = 1e-10,
                   max_rank: float = math.inf) -> List[torch.Tensor]:
    coren, coren_prev = cores[n], cores[n - 1]
    rL, In, K, rR = coren.shape
    rPrev, Iprev, Jprev, rL_chk = coren_prev.shape
    assert rL == rL_chk
    assert Jprev == 1, "Expected singleton physical column index in non-active core"

    M = coren.permute(0, 2, 1, 3).reshape(rL * K, In * rR)
    U, S, Vh = torch.linalg.svd(M, full_matrices=False)

    if tol != 0:
        keep = S > tol * S[0]
        U, S, Vh = U[:, keep], S[keep], Vh[keep, :]
    if max_rank != math.inf:
        r_new = min(S.shape[0], int(max_rank))
        U, S, Vh = U[:, :r_new], S[:r_new], Vh[:r_new, :]
    rank_new = U.shape[1]

    US = U @ torch.diag(S.to(U.dtype))
    US_sh = US.reshape(rL, K, rank_new)
    coren_prev_new = torch.einsum('pil,lka->pika', coren_prev[:, :, 0, :], US_sh)
    coren_new = Vh.reshape(rank_new, In, 1, rR)
    return cores[:n - 1] + [coren_prev_new, coren_new] + cores[n + 1:]

In [5]:
def effective_ops_batch(n: int, cores: List[torch.Tensor], Eb: torch.Tensor) -> torch.Tensor:
    """Batched replacement for the original `Eeffm_n`.

    Args:
        n: core index being solved for.
        cores: current model TT cores (each (rL, d, m, rR); m==1 except at the
            single "active" core, which momentarily carries the extra K leg).
        Eb: (B, order, d, d) tensor -- one local Hermitian matrix per site per
            sample in the batch (a batched stack of product/POVM operators).

    Returns:
        (B, d_row, d_col) tensor of effective operator matrices, one per
        sample, where d_row = rU_n * n_k * rU_{n+1}-ish and d_col matches.
    """
    order = len(cores)
    B = Eb.shape[0]
    Vc = [c[:, :, 0, :] for c in cores]  # (rL, In, rR), shared across the batch

    # Left stack: (B, a=rV_n, d=rU_n)
    stack_L = torch.ones((B, 1, 1), dtype=Eb.dtype, device=Eb.device)
    for i in range(1, n + 1):
        V = Vc[i - 1]
        Op = Eb[:, i - 1]
        Uconj = Vc[i - 1].conj()
        stack_L = torch.einsum('bad,ajc,bkj,dke->bce', stack_L, V, Op, Uconj)

    # Right stack: (B, a=rV_{n+1}, d=rU_{n+1})
    stack_R = torch.ones((B, 1, 1), dtype=Eb.dtype, device=Eb.device)
    for i in range(order - 2, n - 1, -1):
        Uconj = Vc[i + 1].conj()
        Op = Eb[:, i + 1]
        V = Vc[i + 1]
        stack_R = torch.einsum('dkg,bcg,bkj,ajc->bad', Uconj, stack_R, Op, V)

    Opn = Eb[:, n]  # (B, k, j)
    # rows = (rU_n, phys, rU_{n+1}); cols = (rV_n, phys, rV_{n+1})
    eff = torch.einsum('bad,bkj,bfg->bdkgajf', stack_L, Opn, stack_R)
    a, d2 = stack_L.shape[1], stack_L.shape[2]
    k, j = Opn.shape[1], Opn.shape[2]
    f, g = stack_R.shape[1], stack_R.shape[2]
    return eff.reshape(B, d2 * k * g, a * j * f)


def get_probability_batch(cores_a: List[torch.Tensor], Eb: torch.Tensor) -> torch.Tensor:
    """Batched replacement for the original `get_probability_batch`.

    Unlike `effective_ops_batch` (which slices index 0 of any trivial col
    leg, matching the original `_left_interface`/`_right_interface`), this
    function uses the FULL (unsliced) cores and sums over the column block --
    matching the original's `ttmulcores` + `cores2ten` + `trace` pipeline,
    which computes Tr(A^H E A) = sum_k <A_k|E|A_k> when one core carries a
    non-trivial K column ("rank") leg. Both reformulations were validated
    against the original (unbatched) implementations.

    Args:
        cores_a: model TT cores.
        Eb: (B, order, d, d) tensor of local Hermitian operator matrices.

    Returns:
        (B,) real tensor of predicted Born probabilities Tr(A^H E A).
    """
    B = Eb.shape[0]
    stack = torch.ones((B, 1, 1), dtype=Eb.dtype, device=Eb.device)
    for i in range(len(cores_a)):
        V = cores_a[i]            # full core, NOT sliced: (rL, In, m_i, rR)
        Op = Eb[:, i]
        Uconj = cores_a[i].conj()
        stack = torch.einsum('bad,ajmc,bkj,dkme->bce', stack, V, Op, Uconj)
    return stack[:, 0, 0].real


def ops_list_to_batch(Em_list, device, dtype) -> torch.Tensor:
    """Stack a (legacy-style) list of per-sample operator lists -- each a
    list of (1,d,d,1) site matrices -- into a single (B, order, d, d) tensor.
    Use this once when building a dataset; downstream code should keep
    operators in this batched tensor form."""
    order = len(Em_list[0])
    d = Em_list[0][0].shape[1]
    out = torch.empty((len(Em_list), order, d, d), dtype=dtype, device=device)
    for b, Em in enumerate(Em_list):
        for s in range(order):
            out[b, s] = Em[s][0, :, :, 0]
    return out

QST Measurements

In [6]:
class MeasurementDataset:
    """Generates synthetic Born-rule measurement data for a TT/MPO ground-truth
    state `cores_a`. All operator batches are stored as (B, order, d, d) torch
    tensors (rather than nested Python lists) so probability generation and
    later training both use the batched, GPU-resident kernels above."""

    def __init__(self, cores_a: List[torch.Tensor], N: int, measurement_type: str = "Rn",
                 device=None, dtype=None, seed: Optional[int] = None):
        self.cores_a = cores_a
        self.N = N
        self.measurement_type = measurement_type
        self.device = device or cores_a[0].device
        self.dtype = dtype or cores_a[0].dtype
        self.gen = torch.Generator(device="cpu")
        if seed is not None:
            self.gen.manual_seed(seed)

        self.I = torch.eye(2, dtype=self.dtype, device=self.device)
        self.X = torch.tensor([[0, 1], [1, 0]], dtype=self.dtype, device=self.device)
        self.Y = torch.tensor([[0, -1j], [1j, 0]], dtype=self.dtype, device=self.device)
        self.Z = torch.tensor([[1, 0], [0, -1]], dtype=self.dtype, device=self.device)

        self._build_local_povm()
        self.train_data: Optional[Tuple[torch.Tensor, torch.Tensor]] = None
        self.test_data: Optional[Tuple[torch.Tensor, torch.Tensor]] = None

    def _build_local_povm(self):
        I, X, Y, Z = self.I, self.X, self.Y, self.Z
        if self.measurement_type == "Pauli":
            self.local_povm = torch.stack([0.5 * (I + Z), 0.5 * (I - Z), 0.5 * (I + X), 0.5 * (I + Y)])
        elif self.measurement_type == "Tetra":
            s = torch.tensor([
                [0, 0, 1],
                [2 * math.sqrt(2) / 3, 0, -1 / 3],
                [-math.sqrt(2) / 3, math.sqrt(2 / 3), -1 / 3],
                [-math.sqrt(2) / 3, -math.sqrt(2 / 3), -1 / 3],
            ], dtype=torch.float64, device=self.device)
            paulis = torch.stack([X, Y, Z])  # (3,2,2)
            self.local_povm = 0.25 * (I.unsqueeze(0) + torch.einsum('rp,pij->rij', s.to(self.dtype), paulis))
        elif self.measurement_type == "SIC":
            v = torch.tensor([[1, 1, 1], [1, -1, -1], [-1, 1, -1], [-1, -1, 1]],
                              dtype=torch.float64, device=self.device) / math.sqrt(3)
            v = v.to(self.dtype)
            ops = []
            for r in v:
                E = 0.5 * (I + r[0] * X + r[1] * Y + r[2] * Z)
                ops.append(E / 2)
            self.local_povm = torch.stack(ops)
        elif self.measurement_type == "Rn":
            self.local_povm = None
        else:
            raise ValueError(f"Unknown POVM type: {self.measurement_type}")

    def _Rn(self, alpha, phi, theta):
        nx = math.sin(alpha) * math.cos(phi)
        ny = math.sin(alpha) * math.sin(phi)
        nz = math.cos(alpha)
        return (math.cos(theta / 2) * self.I
                - 1j * math.sin(theta / 2) * (nx * self.X + ny * self.Y + nz * self.Z))

    def _local_Rn_projector(self, alpha, phi, theta):
        U = self._Rn(alpha, phi, theta)
        psi = U @ torch.tensor([1, 0], dtype=self.dtype, device=self.device)
        return torch.outer(psi, psi.conj())

    def _sample_batch(self, m: int) -> torch.Tensor:
        """Sample m product-operator measurements -> (m, N, d, d) tensor."""
        d = 2
        ops = torch.empty((m, self.N, d, d), dtype=self.dtype, device=self.device)
        if self.measurement_type == "Rn":
            for b in range(m):
                for s in range(self.N):
                    alpha = math.pi * torch.rand(1, generator=self.gen).item()
                    phi = 2 * math.pi * torch.rand(1, generator=self.gen).item()
                    theta = 2 * math.pi * torch.rand(1, generator=self.gen).item()
                    ops[b, s] = self._local_Rn_projector(alpha, phi, theta)
        else:
            n_povm = self.local_povm.shape[0]
            idx = torch.randint(0, n_povm, (m, self.N), generator=self.gen)
            ops = self.local_povm[idx]
        return ops

    def gen_data(self, m_train: int, m_test: int):
        Eb_train = self._sample_batch(m_train)
        y_train = get_probability_batch(self.cores_a, Eb_train)
        self.train_data = (Eb_train, y_train)

        Eb_test = self._sample_batch(m_test)
        y_test = get_probability_batch(self.cores_a, Eb_test)
        self.test_data = (Eb_test, y_test)


class DataLoader:
    """Batched loader over an (Eb, y) tuple of tensors (replaces the original
    list-of-tuples loader, since data now lives as stacked GPU tensors)."""

    def __init__(self, data: Tuple[torch.Tensor, torch.Tensor], batch_size: int = 32, shuffle: bool = True):
        self.Eb, self.y = data
        self.batch_size = batch_size
        self.shuffle = shuffle
        self.n = self.Eb.shape[0]

    def __iter__(self):
        self.indices = torch.randperm(self.n) if self.shuffle else torch.arange(self.n)
        self.ptr = 0
        return self

    def __next__(self):
        if self.ptr >= self.n:
            raise StopIteration
        idx = self.indices[self.ptr:self.ptr + self.batch_size]
        self.ptr += self.batch_size
        return self.Eb[idx], self.y[idx]

    def __len__(self):
        return (self.n + self.batch_size - 1) // self.batch_size


States

In [7]:
def ghz_mps(N: int, extra_dim: bool = True, device=None, dtype=None) -> List[torch.Tensor]:
    device, dtype = device or default_device(), dtype or torch.float64
    A = []
    A0 = torch.zeros((1, 2, 2), dtype=dtype, device=device)
    A0[0, 0, 0], A0[0, 1, 1] = 1.0, 1.0
    A.append(A0)
    for _ in range(N - 2):
        Ak = torch.zeros((2, 2, 2), dtype=dtype, device=device)
        Ak[0, 0, 0], Ak[1, 1, 1] = 1.0, 1.0
        A.append(Ak)
    AN = torch.zeros((2, 2, 1), dtype=dtype, device=device)
    AN[0, 0, 0], AN[1, 1, 0] = 1.0, 1.0
    A.append(AN)
    if extra_dim:
        A = [a.unsqueeze(2) for a in A]
    return A


def w_mps(N: int, extra_dim: bool = True, device=None, dtype=None) -> List[torch.Tensor]:
    device, dtype = device or default_device(), dtype or torch.float64
    A = []
    A0 = torch.zeros((1, 2, 2), dtype=dtype, device=device)
    A0[0, 0, 0], A0[0, 1, 1] = 1.0, 1.0
    A.append(A0)
    for _ in range(N - 2):
        Ak = torch.zeros((2, 2, 2), dtype=dtype, device=device)
        Ak[0, 0, 0], Ak[1, 0, 1], Ak[0, 1, 1] = 1.0, 1.0, 1.0
        A.append(Ak)
    AN = torch.zeros((2, 2, 1), dtype=dtype, device=device)
    AN[0, 1, 0], AN[1, 0, 0] = 1.0, 1.0
    A.append(AN)
    if extra_dim:
        A = [a.unsqueeze(2) for a in A]
    return A

Optimization

In [8]:
# ===========================================================================
# 6. Local solver -- trace objective (used inside ALS on effective operators)
#    Objective/gradient/Hessian-vector-product are vectorized over the whole
#    batch of effective operators with `einsum` (no Python loop over samples).
# ===========================================================================

def pack_complex(A: torch.Tensor) -> torch.Tensor:
    """Flatten a complex (d,r) tensor into a real vector [Re; Im]."""
    real_dtype = torch.float64 if A.dtype == torch.complex128 else torch.float32
    return torch.cat([A.real.reshape(-1).to(real_dtype), A.imag.reshape(-1).to(real_dtype)])


def unpack_complex(x: torch.Tensor, d: int, r: int, dtype=torch.complex128) -> torch.Tensor:
    n = d * r
    Are = x[:n].reshape(d, r)
    Aim = x[n:].reshape(d, r)
    return torch.complex(Are, Aim).to(dtype)


def objective(x: torch.Tensor, Ob: torch.Tensor, y: torch.Tensor, d: int, r: int) -> torch.Tensor:
    """Ob: (B,d,d) batch of effective operators. Vectorized: one einsum, no Python loop."""
    A = unpack_complex(x, d, r, dtype=Ob.dtype)
    ri = torch.einsum('da,bde,ea->b', A.conj(), Ob, A).real - y
    return torch.sum(ri * ri)


def gradient(x: torch.Tensor, Ob: torch.Tensor, y: torch.Tensor, d: int, r: int) -> torch.Tensor:
    A = unpack_complex(x, d, r, dtype=Ob.dtype)
    ri = torch.einsum('da,bde,ea->b', A.conj(), Ob, A).real - y
    symO = Ob + Ob.conj().transpose(-1, -2)
    G = 2 * torch.einsum('b,bde,ea->da', ri.to(Ob.dtype), symO, A)
    return pack_complex(G)


def hessian_vec(x: torch.Tensor, v: torch.Tensor, Ob: torch.Tensor, y: torch.Tensor,
                 d: int, r: int) -> torch.Tensor:
    A = unpack_complex(x, d, r, dtype=Ob.dtype)
    Delta = unpack_complex(v, d, r, dtype=Ob.dtype)
    ri = torch.einsum('da,bde,ea->b', A.conj(), Ob, A).real - y
    symO = Ob + Ob.conj().transpose(-1, -2)
    symOD = torch.einsum('bde,ea->bda', symO, Delta)
    symOA = torch.einsum('bde,ea->bda', symO, A)
    term1 = 2 * torch.einsum('b,bda->da', ri.to(Ob.dtype), symOD)
    inner = torch.einsum('da,bda->b', Delta.conj(), symOA).real
    term2 = 2 * torch.einsum('b,bda->da', inner.to(Ob.dtype), symOA)
    return pack_complex(term1 + term2)


def minimize_trace(A0: torch.Tensor, Ob: torch.Tensor, y: torch.Tensor,
                    max_iter: int = 100, method: str = 'L-BFGS-B') -> Tuple[torch.Tensor, dict]:
    """Drop-in GPU replacement for the original scipy-based `minimize_trace`.
    `Ob` is the (B,d,d) batch of effective operators (already stacked into a
    tensor, e.g. via `effective_ops_batch`)."""
    d, r = A0.shape
    x0 = pack_complex(A0)
    obj = lambda x: objective(x, Ob, y, d, r)
    grad = lambda x: gradient(x, Ob, y, d, r)

    if method == 'Newton-CG':
        hvp = lambda x, v: hessian_vec(x, v, Ob, y, d, r)
        x_opt, info = newton_cg_minimize(x0, obj, grad, hvp, max_iter=max_iter, xtol=1e-10)
    elif method == 'L-BFGS-B':
        x_opt, info = lbfgs_minimize(x0, obj, grad, max_iter=max_iter, ftol=1e-12)
    else:
        raise ValueError("Unknown method")

    return unpack_complex(x_opt, d, r, dtype=Ob.dtype), info


# ===========================================================================
# 7. Local solver -- normalized objective (small non-TT sanity check, cell 14)
# ===========================================================================

def objective_norm(x: torch.Tensor, Ob: torch.Tensor, y: torch.Tensor, d: int, r: int) -> torch.Tensor:
    A = unpack_complex(x, d, r, dtype=Ob.dtype)
    nrm = torch.linalg.norm(A)
    Ahat = A if nrm < 1e-14 else A / nrm
    ri = torch.einsum('da,bde,ea->b', Ahat.conj(), Ob, Ahat).real - y
    return torch.sum(ri * ri)


def gradient_norm(x: torch.Tensor, Ob: torch.Tensor, y: torch.Tensor, d: int, r: int) -> torch.Tensor:
    A = unpack_complex(x, d, r, dtype=Ob.dtype)
    nrm = torch.linalg.norm(A)
    if nrm < 1e-14:
        return torch.zeros_like(x)
    Ahat = A / nrm
    ri = torch.einsum('da,bde,ea->b', Ahat.conj(), Ob, Ahat).real - y
    G = 4 * torch.einsum('b,bde,ea->da', ri.to(Ob.dtype), Ob, Ahat)
    alpha = torch.einsum('da,da->', Ahat.conj(), G).real
    G_tan = (G - alpha.to(Ob.dtype) * Ahat) / nrm
    return pack_complex(G_tan)


def hessp_norm(x: torch.Tensor, v: torch.Tensor, Ob: torch.Tensor, y: torch.Tensor,
               d: int, r: int, damping: float = 1e-6, eps: float = 1e-6) -> torch.Tensor:
    """Finite-difference HVP of the normalized gradient (matches the original,
    which also used central differences here rather than an exact analytic HVP)."""
    grad_plus = gradient_norm(x + eps * v, Ob, y, d, r)
    grad_minus = gradient_norm(x - eps * v, Ob, y, d, r)
    Hv = (grad_plus - grad_minus) / (2 * eps)
    return Hv + damping * v


def minimize_norm(A0: torch.Tensor, Ob: torch.Tensor, y: torch.Tensor, method: str = 'L-BFGS-B',
                   max_iter: int = 200, damping: float = 1e-6) -> Tuple[torch.Tensor, dict]:
    d, r = A0.shape
    x0 = pack_complex(A0)
    obj = lambda x: objective_norm(x, Ob, y, d, r)
    grad = lambda x: gradient_norm(x, Ob, y, d, r)

    if method == 'Newton-CG':
        hvp = lambda x, v: hessp_norm(x, v, Ob, y, d, r, damping=damping) - damping * v
        x_opt, info = newton_cg_minimize(x0, obj, grad, hvp, max_iter=max_iter, xtol=1e-10, damping=damping)
    elif method == 'L-BFGS-B':
        x_opt, info = lbfgs_minimize(x0, obj, grad, max_iter=max_iter, ftol=1e-12)
    elif method == 'TR':
        try:
            x_opt, info = minimize_norm(A0, Ob, y, method='Newton-CG', max_iter=max_iter, damping=damping)
            if not info.get('converged', False):
                raise RuntimeError("Newton-CG did not converge")
        except Exception:
            x_opt, info = minimize_norm(A0, Ob, y, method='L-BFGS-B', max_iter=max_iter)
            return x_opt, info
        A_opt = x_opt
        nrm = torch.linalg.norm(A_opt)
        if nrm > 1e-14:
            A_opt = A_opt / nrm
        return A_opt, info
    else:
        raise ValueError("Unknown method")

    A_opt = unpack_complex(x_opt, d, r, dtype=Ob.dtype)
    nrm = torch.linalg.norm(A_opt)
    if nrm > 1e-14:
        A_opt = A_opt / nrm
    return A_opt, info


# ===========================================================================
# 8. Custom GPU-resident optimizers (replace scipy.optimize.minimize)
#
# scipy's L-BFGS-B / Newton-CG only operate on CPU numpy arrays. These two
# functions implement the same family of unconstrained smooth-optimization
# algorithms directly on torch tensors (any device), using only basic vector
# ops (dot, norm, axpy) -- so the whole ALS inner loop stays GPU-resident with
# no CPU synchronization.
# ===========================================================================

def _backtracking_line_search(fun, x, f0, g, d, c1=1e-4, max_ls=30, min_step=1e-12):
    """Armijo backtracking line search on a flat real tensor `x`."""
    gd = torch.dot(g, d)
    if gd.item() >= 0:  # not a descent direction -> fall back to steepest descent
        d = -g
        gd = torch.dot(g, d)
    t = 1.0
    x_new, f_new = x, f0
    for _ in range(max_ls):
        x_new = x + t * d
        f_new = fun(x_new)
        if torch.isfinite(f_new) and f_new.item() <= f0 + c1 * t * gd.item():
            return t, x_new, f_new, d
        t *= 0.5
        if t < min_step:
            break
    x_new = x + t * d
    f_new = fun(x_new)
    return t, x_new, f_new, d


def lbfgs_minimize(x0: torch.Tensor, fun, grad, max_iter: int = 100,
                    ftol: float = 1e-12, gtol: float = 1e-10, history_size: int = 10) -> Tuple[torch.Tensor, dict]:
    """Limited-memory BFGS (unconstrained), GPU-resident. `fun(x)->scalar tensor`,
    `grad(x)->vector tensor`. Replaces scipy's L-BFGS-B (no bounds are used in
    the original notebook, so this unconstrained variant is equivalent)."""
    x = x0.clone()
    f = fun(x)
    g = grad(x)

    s_list, y_list, rho_list = [], [], []
    n_iter, converged = 0, False

    for it in range(max_iter):
        n_iter = it + 1
        if torch.linalg.norm(g).item() < gtol:
            converged = True
            break

        q = g.clone()
        alphas = []
        for i in range(len(s_list) - 1, -1, -1):
            a_i = rho_list[i] * torch.dot(s_list[i], q)
            alphas.append(a_i)
            q = q - a_i * y_list[i]
        alphas.reverse()
        if s_list:
            sy = torch.dot(s_list[-1], y_list[-1])
            yy = torch.dot(y_list[-1], y_list[-1])
            gamma = (sy / yy) if yy.item() > 1e-18 else torch.tensor(1.0, dtype=x.dtype, device=x.device)
        else:
            gamma = torch.tensor(1.0, dtype=x.dtype, device=x.device)
        r = gamma * q
        for i in range(len(s_list)):
            beta_i = rho_list[i] * torch.dot(y_list[i], r)
            r = r + (alphas[i] - beta_i) * s_list[i]
        d = -r

        t, x_new, f_new, d = _backtracking_line_search(fun, x, f.item(), g, d)
        g_new = grad(x_new)

        s = x_new - x
        yv = g_new - g
        sy = torch.dot(s, yv)
        if sy.item() > 1e-12:
            s_list.append(s)
            y_list.append(yv)
            rho_list.append(1.0 / sy)
            if len(s_list) > history_size:
                s_list.pop(0); y_list.pop(0); rho_list.pop(0)

        f_change = abs(f_new.item() - f.item())
        x, f, g = x_new, f_new, g_new
        if f_change <= ftol * max(1.0, abs(f.item())):
            converged = True
            break

    return x, {"fun": f.item(), "nit": n_iter, "success": True, "converged": converged}


def _cg_solve(hvp, b: torch.Tensor, max_iter: Optional[int] = None, tol: float = 1e-6) -> torch.Tensor:
    """Steihaug-style truncated CG solving H p = b; bails out on negative curvature."""
    n = b.shape[0]
    if max_iter is None:
        max_iter = n
    x = torch.zeros_like(b)
    r = b - hvp(x)
    p = r.clone()
    rs_old = torch.dot(r, r)
    bnorm = torch.linalg.norm(b)
    if bnorm.item() < 1e-300 or torch.sqrt(rs_old).item() <= tol * max(1.0, bnorm.item()):
        return x
    for i in range(max_iter):
        Hp = hvp(p)
        pHp = torch.dot(p, Hp)
        if pHp.item() <= 1e-14:
            return x if i > 0 else b
        alpha = rs_old / pHp
        x = x + alpha * p
        r = r - alpha * Hp
        rs_new = torch.dot(r, r)
        if torch.sqrt(rs_new).item() <= tol * max(1.0, bnorm.item()):
            break
        p = r + (rs_new / rs_old) * p
        rs_old = rs_new
    return x


def newton_cg_minimize(x0: torch.Tensor, fun, grad, hessp, max_iter: int = 100, xtol: float = 1e-10,
                        cg_tol: float = 1e-6, cg_max_iter: Optional[int] = None,
                        damping: float = 1e-6) -> Tuple[torch.Tensor, dict]:
    """Truncated Newton-CG, GPU-resident. `hessp(x, v) -> H(x) @ v` is the exact
    analytic Hessian-vector product (no autograd). `damping` adds a small
    Levenberg-Marquardt-style ridge for stability, matching the original."""
    x = x0.clone()
    n_iter, converged = 0, False
    f = fun(x)

    def damped_hessp(xv, v):
        return hessp(xv, v) + damping * v

    for it in range(max_iter):
        n_iter = it + 1
        g = grad(x)
        if torch.linalg.norm(g).item() < 1e-12:
            converged = True
            break

        p = _cg_solve(lambda v: damped_hessp(x, v), -g, max_iter=cg_max_iter, tol=cg_tol)
        t, x_new, f_new, _ = _backtracking_line_search(fun, x, f.item(), g, p)
        step_norm = torch.linalg.norm(x_new - x).item()
        x, f = x_new, f_new
        if step_norm < xtol:
            converged = True
            break

    return x, {"fun": f.item(), "nit": n_iter, "success": True, "converged": converged}

In [9]:
def als_ttqst(init_cores_a: List[torch.Tensor], train_loader: DataLoader, nsweeps: int = 3,
              max_iter_inner: int = 100, max_rank: int = 50, verbose: bool = True) -> Tuple[List[torch.Tensor], List[float]]:
    cores = copy_cores(init_cores_a)
    order = len(cores)
    loss_history = []
    sweep_order = list(range(order - 1)) + list(range(order - 1, 0, -1))

    for swp in range(nsweeps):
        incr = True
        core_loss = []
        for n in sweep_order:
            if n == order - 1:
                incr = False
            if verbose:
                print('Optimizing core #', n)

            batch_loss = []
            for Eb, y in train_loader:
                p_pred = get_probability_batch(cores, Eb)
                batch_loss.append((torch.linalg.norm(y - p_pred) / torch.linalg.norm(y)).item())

                Ob = effective_ops_batch(n, cores, Eb)

                r_prev, n_k, m_k, r_next = cores[n].shape
                An0 = cores[n].permute(0, 1, 3, 2).reshape(-1, m_k)

                method = 'L-BFGS-B' if swp <= 2 else 'Newton-CG'
                new_core, res = minimize_trace(An0, Ob, y, max_iter=max_iter_inner, method=method)
                cores[n] = new_core.reshape(r_prev, n_k, r_next, m_k).permute(0, 1, 3, 2)

            core_loss.append(sum(batch_loss) / len(batch_loss))
            if verbose:
                print('mean loss over batches', sum(core_loss) / len(core_loss))

            if incr:
                cores = block2rightbtt(cores, n, max_rank=max_rank)
                fnorm = torch.linalg.norm(cores[n + 1]).item()
                if fnorm > 1:
                    cores[n + 1] = cores[n + 1] / fnorm
            else:
                cores = block2leftbtt(cores, n, max_rank=max_rank)
                fnorm = torch.linalg.norm(cores[n - 1]).item()
                if fnorm > 1:
                    cores[n - 1] = cores[n - 1] / fnorm

        loss_history.append(sum(core_loss) / len(core_loss))
        if verbose:
            print("===================================")
            print(f"Sweep loss : {loss_history[-1]:.10f}")
            print("===================================")

    return cores, loss_history

In [11]:
N: int = 6
K: int = 2
max_rank: int = 3
nsweeps: int = 2
max_iter_inner: int = 100
measurement_type: str = "Tetra"
seed_true: int = 43
seed_init: int = 49
batch_size: int = 256
device = default_device()
dtype = torch.complex128
print(f"Running on device: {device}")

row_dims = [2] * N
col_dims = [1] * N
col_dims[0] = K
tt_rank = [1] + [3] * (N - 1) + [1]

# ground-truth state: rho = A A^H, A a random block-TT factor
cores_a = init_ttm_cores(row_dims, col_dims, tt_rank, 1, seed_true, complex_output=True,
                          device=device, dtype=dtype)
cores_a = ortho_right(cores_a, 1e-10)
cores_a[0] = cores_a[0] / torch.linalg.norm(cores_a[0])
cores_ah = transpose_cores(cores_a, overwrite=False, conjugate=True)
cores_true = ttmulcores(cores_a, cores_ah)
cores_true = ortho_right(cores_true)
Mat = cores2ten(cores_true, True)


# random initial guess for the model to be learned
# cores_a0 = init_ttm_cores(row_dims, col_dims, tt_rank, 1, seed_init, complex_output=True,
#                             device=device, dtype=dtype)
cores_a0 = ghz_mps(N, True, device=device, dtype=dtype)
cores_a0 = ortho_right(cores_a0)
cores_a0[0] = cores_a0[0] / torch.linalg.norm(cores_a0[0])
cores_ah0 = transpose_cores(cores_a0, overwrite=False, conjugate=True)
Mrec0 = cores2ten(ttmulcores(cores_a0, cores_ah0), True)
print("initial guess: fidelity =", fidelity(Mat.cpu().numpy(), Mrec0.cpu().numpy()), " tracedist =", trace_dist(Mat.cpu().numpy(), Mrec0.cpu().numpy()))

# measurement dataset
train_samples = int(5 * N * math.log(N) * 4 * K * max_rank ** 2)
ds = MeasurementDataset(cores_a, N, measurement_type=measurement_type, device=device, dtype=dtype)
ds.gen_data(train_samples, train_samples // 4)
loader = DataLoader(ds.train_data, batch_size=batch_size, shuffle=False)

t0 = time.time()
Aopt, cost = als_ttqst(cores_a0, loader, nsweeps=nsweeps, max_iter_inner=max_iter_inner,
                        max_rank=max_rank, verbose=1)
print(f"Training time: {time.time() - t0:.2f}s")

Aopt_h = transpose_cores(Aopt, overwrite=False, conjugate=True)
Mrec = cores2ten(ttmulcores(Aopt, Aopt_h), True)

print("recovered: fidelity =", fidelity(Mat.cpu().numpy(), Mrec.cpu().numpy()),"  tracedist =", trace_dist(Mat.cpu().numpy(), Mrec.cpu().numpy()))

test_loader = DataLoader(ds.test_data, batch_size=batch_size, shuffle=False)
rel_errs = []
for Eb, y in test_loader:
    p_pred = get_probability_batch(Aopt, Eb)
    rel_errs.append((torch.linalg.norm(y - p_pred) / torch.linalg.norm(y)).item())
print("test relative error:", sum(rel_errs) / len(rel_errs))

Running on device: cuda
initial guess: fidelity = 0.08474871797368584  tracedist = 0.9952470797414468
Optimizing core # 0
mean loss over batches 0.9276001057790653
Optimizing core # 1
mean loss over batches 0.9198011059223028
Optimizing core # 2
mean loss over batches 0.9158219097342729
Optimizing core # 3
mean loss over batches 0.9132052120871463
Optimizing core # 4
mean loss over batches 0.9103998695322797
Optimizing core # 5
mean loss over batches 0.9071451202774963
Optimizing core # 4
mean loss over batches 0.9035061438736758
Optimizing core # 3
mean loss over batches 0.8998835412215154
Optimizing core # 2
mean loss over batches 0.8965535100408277
Optimizing core # 1
mean loss over batches 0.8932825933569664
Sweep loss : 0.8932825934
Optimizing core # 0
mean loss over batches 0.8595769595327439
Optimizing core # 1
mean loss over batches 0.8574920192700067
Optimizing core # 2
mean loss over batches 0.854710700668544
Optimizing core # 3
mean loss over batches 0.8520614250771241
Optim